# Constrained Decoding for Autoregressive Models

Autoregressive language models assign a probability to a token sequence one token at a time. For a completion $y = (y_1, \ldots, y_n)$ after a prompt $x$,

$$\log P(y \mid x) = \sum_{t=1}^{n} \log P(y_t \mid x, y_{<t}).$$

This notebook uses that factorization to choose among a finite list of values for each field. It encodes the shared prompt once, keeps the model's KV cache, and scores every allowed value as a continuation. The chosen values are then assembled into a Python dictionary.

That distinction matters: this is **constrained classification**, not grammar-constrained generation of an entire JSON document. The final dictionary is valid because ordinary Python code constructs it. The model never generates punctuation, field order, or arbitrary JSON text.

Each candidate is scored through its closing quote, so multi-token choices are treated consistently. The sequence log-probabilities are normalized over the allowed values:

$$q(c_k \mid x) = \frac{\exp(s_k)}{\sum_j \exp(s_j)}, \qquad s_k = \log P(\text{value } c_k \text{ and closing quote} \mid x).$$

These normalized values are useful for ranking candidates, but they are not calibrated real-world confidence scores. They are conditional on the candidate list and the wording of the prompt.


## 1. Setup

The example uses `Qwen/Qwen2.5-1.5B-Instruct`. With `device_map="auto"`, inputs need to follow the model's embedding device rather than a separate CUDA check.


In [ ]:
import copy
import json
from dataclasses import dataclass
from typing import Any, Dict, List

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer


CHECKPOINT = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT,
    torch_dtype="auto",
    device_map="auto",
)
model.eval()

input_device = model.get_input_embeddings().weight.device
print(f"Model input device: {input_device}")


## 2. Describe the fields

A field has a name, a short description, and a finite list of allowed values. Validation trims accidental whitespace and rejects empty or duplicate values. The dictionary key is checked against `FieldDefinition.name` later, since a mismatch would make the prompt and result disagree.


In [ ]:
@dataclass
class FieldDefinition:
    name: str
    description: str
    choices: List[str]

    def __post_init__(self) -> None:
        self.name = self.name.strip()
        self.description = self.description.strip()

        if not self.name:
            raise ValueError("Field name cannot be empty.")
        if not self.description:
            raise ValueError(f"Field {self.name!r} needs a description.")
        if not self.choices:
            raise ValueError(
                f"Field {self.name!r} must contain at least one choice."
            )

        cleaned_choices = [str(choice).strip() for choice in self.choices]
        if any(not choice for choice in cleaned_choices):
            raise ValueError(f"Field {self.name!r} contains an empty choice.")
        if len(set(cleaned_choices)) != len(cleaned_choices):
            raise ValueError(f"Field {self.name!r} contains duplicate choices.")

        self.choices = cleaned_choices


schema: Dict[str, FieldDefinition] = {
    "weather": FieldDefinition(
        name="weather",
        description="The forecast weather condition.",
        choices=["sunny", "rainy", "fair", "cold"],
    ),
    "gear": FieldDefinition(
        name="gear",
        description="The most useful item to bring.",
        choices=["umbrella", "jacket", "shades"],
    ),
}


## 3. Build one shared prompt

The prompt contains the input and every allowed value, then stops at the opening brace. Each field is evaluated as an alternative continuation from that point. Fields therefore do not condition on one another; this is suitable for independent classifications, but not for schemas where one answer changes another field's meaning or allowed values.

KV-cache reuse is exact only when tokenizing `prefix + continuation` leaves the prefix token IDs unchanged. Byte-pair tokenizers can merge text across a boundary. Silently tokenizing the continuation on its own would then score a different sequence, so `continuation_token_ids` checks the boundary and fails clearly if it is unstable.


In [ ]:
def validate_schema(schema: Dict[str, FieldDefinition]) -> None:
    if not schema:
        raise ValueError("Schema must contain at least one field.")

    for field_name, field in schema.items():
        if field_name != field.name:
            raise ValueError(
                f"Schema key {field_name!r} does not match field name "
                f"{field.name!r}."
            )


def build_schema_prompt(
    context: str,
    schema: Dict[str, FieldDefinition],
) -> str:
    validate_schema(schema)

    field_lines = []
    for field_name, field in schema.items():
        choices = ", ".join(json.dumps(c) for c in field.choices)
        field_lines.append(
            f"- {json.dumps(field_name)}: {field.description} "
            f"Allowed values: {choices}."
        )

    messages = [
        {
            "role": "system",
            "content": (
                "Choose values only from the allowed lists. "
                "Use the input as evidence and do not add an explanation."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Input:\n{context.strip()}\n\nFields:\n"
                + "\n".join(field_lines)
                + "\n\nReturn one JSON object containing every field."
            ),
        },
    ]

    chat_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    return chat_prompt + "{"


def continuation_token_ids(prefix: str, full_text: str) -> List[int]:
    if not full_text.startswith(prefix):
        raise ValueError("full_text must begin with prefix.")

    prefix_ids = tokenizer.encode(prefix, add_special_tokens=False)
    full_ids = tokenizer.encode(full_text, add_special_tokens=False)
    if full_ids[: len(prefix_ids)] != prefix_ids:
        raise ValueError(
            "The tokenizer merged tokens across a cache boundary. "
            "Change the separator before the continuation and try again."
        )

    return full_ids[len(prefix_ids) :]


## 4. Score complete candidate values

  For a field, the model first processes a suffix such as `
"weather": "` on top of the shared prompt cache. Its final logits score the first token of every candidate. The rest of each candidate, including the closing quote, is scored with teacher forcing. This avoids the main problem with first-token shortcuts: two candidates can share a first token, and a unique first token is not a probability for the complete value.

  Some Transformers cache implementations are mutable. A deep copy is passed into each branch so scoring one candidate cannot extend the cache seen by the next candidate. This is simple and safe for a teaching example, though a production implementation would usually batch candidates or use an explicitly branchable cache.


In [ ]:
def make_input_tensor(token_ids: List[int]) -> torch.Tensor:
    return torch.tensor(
        [token_ids],
        dtype=torch.long,
        device=input_device,
    )


def token_log_probability(
    logits: torch.Tensor,
    token_id: int,
    temperature: float,
) -> torch.Tensor:
    log_probs = F.log_softmax(logits.float() / temperature, dim=-1)
    return log_probs[token_id]


@torch.inference_mode()
def score_field_cached(
    prompt: str,
    prompt_kv: Any,
    field_name: str,
    field: FieldDefinition,
    temperature: float = 1.0,
) -> Dict[str, Any]:
    if temperature <= 0:
        raise ValueError("Temperature must be greater than zero.")

    field_suffix = f'\n  {json.dumps(field_name)}: "'
    field_prefix = prompt + field_suffix
    suffix_ids = continuation_token_ids(prompt, field_prefix)
    if not suffix_ids:
        raise ValueError(f"Field suffix for {field_name!r} produced no tokens.")

    field_outputs = model(
        input_ids=make_input_tensor(suffix_ids),
        past_key_values=copy.deepcopy(prompt_kv),
        use_cache=True,
    )
    first_token_logits = field_outputs.logits[0, -1, :]
    field_kv = field_outputs.past_key_values

    candidate_rows = []
    sequence_scores = []
    forward_passes = 1

    for choice in field.choices:
        # json.dumps supplies escaping and the closing quote. The opening
        # quote is already present in field_suffix.
        choice_tail = json.dumps(choice, ensure_ascii=False)[1:]
        completion_ids = continuation_token_ids(
            field_prefix,
            field_prefix + choice_tail,
        )
        if not completion_ids:
            raise ValueError(f"Choice {choice!r} produced no tokens.")

        token_scores = [
            token_log_probability(
                first_token_logits, completion_ids[0], temperature
            )
        ]

        if len(completion_ids) > 1:
            continuation_outputs = model(
                input_ids=make_input_tensor(completion_ids[:-1]),
                past_key_values=copy.deepcopy(field_kv),
                use_cache=False,
            )
            next_logits = continuation_outputs.logits[0]
            for position, token_id in enumerate(completion_ids[1:]):
                token_scores.append(
                    token_log_probability(
                        next_logits[position], token_id, temperature
                    )
                )
            forward_passes += 1

        sequence_score = torch.stack(token_scores).sum()
        sequence_scores.append(sequence_score)
        candidate_rows.append(
            {
                "choice": choice,
                "token_ids": completion_ids,
                "token_count": len(completion_ids),
                "sequence_log_probability": float(sequence_score.item()),
            }
        )

    probabilities = torch.softmax(torch.stack(sequence_scores), dim=0)
    winner_index = int(probabilities.argmax().item())

    for row, probability in zip(candidate_rows, probabilities):
        row["candidate_probability"] = float(probability.item())

    return {
        "field": field_name,
        "winner": field.choices[winner_index],
        "candidate_probability": float(probabilities[winner_index].item()),
        "candidates": candidate_rows,
        "field_suffix": field_suffix,
        "forward_passes": forward_passes,
    }


## 5. Classify the schema

The shared prompt is encoded once. Every field starts from the same saved state, which makes the field decisions independent given the prompt. The implementation loops over fields for clarity; independence means the work could be scheduled in parallel, but no wall-clock speedup is claimed here.


In [ ]:
@torch.inference_mode()
def classify_schema(
    context: str,
    schema: Dict[str, FieldDefinition],
    temperature: float = 1.0,
) -> Dict[str, Any]:
    if temperature <= 0:
        raise ValueError("Temperature must be greater than zero.")

    prompt = build_schema_prompt(context=context, schema=schema)
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
    if not prompt_ids:
        raise ValueError("The prompt produced no tokens.")

    prompt_outputs = model(
        input_ids=make_input_tensor(prompt_ids),
        use_cache=True,
    )
    prompt_kv = prompt_outputs.past_key_values

    predictions: Dict[str, str] = {}
    details: Dict[str, Any] = {}
    total_forward_passes = 1

    for field_name, field in schema.items():
        field_result = score_field_cached(
            prompt=prompt,
            prompt_kv=prompt_kv,
            field_name=field_name,
            field=field,
            temperature=temperature,
        )
        predictions[field_name] = field_result["winner"]
        details[field_name] = field_result
        total_forward_passes += field_result["forward_passes"]

    return {
        "predictions": predictions,
        "details": details,
        "prompt": prompt,
        "total_forward_passes": total_forward_passes,
    }


## 6. Example

The result includes the selected dictionary and a per-candidate breakdown. No output is stored in this notebook because model revisions, hardware, and package versions can all change the numbers.


In [ ]:
context = (
    "The forecast is 14 degrees Celsius with steady rain. "
    "I need to choose one item to bring."
)

result = classify_schema(
    context=context,
    schema=schema,
    temperature=1.0,
)

print("Selected values:")
print(json.dumps(result["predictions"], indent=2))
print(f"\nForward passes: {result['total_forward_passes']}")

for field_name, field_result in result["details"].items():
    print(f"\n{field_name}:")
    ranked = sorted(
        field_result["candidates"],
        key=lambda row: row["candidate_probability"],
        reverse=True,
    )
    for row in ranked:
        print(
            f"  {row['choice']:<12} "
            f"q={row['candidate_probability']:.4f}  "
            f"log_p={row['sequence_log_probability']:.4f}  "
            f"tokens={row['token_count']}"
        )


## 7. Empirical finding: the trie looked suspicious

Multi-token choices can collide when they begin with the same token. I tried a trie fallback that followed the shared tokens until the choices split. In this test, the prompt said the sky would stay clear and sunny all day. The allowed values were `not sunny`, `not rainy`, and `sunny`. The correct answer was `sunny`, but the trie ranked `not sunny` first.

The two negative choices collided on their first token, `not`, so the fallback extended that shared branch to distinguish `sunny` from `rainy`. The standalone `sunny` choice kept only its first-token score.

The final comparison therefore mixed different amounts of evidence: a partial sequence score for the collision group and a first-token score for the unique branch. That made the result hard to interpret as a clean comparison of the three complete choices. Full-sequence scoring, used above, is one response. The original experiment also tried a practical alternative: put a short label beside each multi-token choice and ask the model to score the labels.

A label experiment has its own failure mode. Language models sometimes favor a familiar answer position or the token `A`, regardless of the mapped content. To check for that, the test below deliberately uses less conventional labels—`Z`, `X`, `W`, and `Q`—and changes the label-to-choice mapping on every run. The useful comparison is the decoded choice, not the winning letter.

In the reported experiment, the semantic winner stayed the same even though its letter changed. That is evidence against a fixed-letter or “always choose A” explanation for that run. It is not a universal proof that labels are unbiased, so the shuffle should be repeated whenever the model, prompt, or choices change.

The observed pattern is easier to see with the mappings written out:

| run | label mapping | winning label | decoded choice |
|---|---|---:|---|
| 1 | `Z = not sunny`, `X = not rainy`, `W = sunny` | `W` | `sunny` |
| 2 | `Q = not rainy`, `W = not sunny`, `Z = sunny` | `Z` | `sunny` |
| 3 | `X = sunny`, `Q = not sunny`, `W = not rainy` | `X` | `sunny` |

The winning token changed from `W` to `Z` to `X`, but each token decoded to `sunny` under its run's mapping. If the model had simply preferred one letter, the semantic answer would have changed when that letter was reassigned. The example therefore supports the content-mapping explanation for these runs, while leaving open the possibility of other prompt or position biases.



In [ ]:
def make_labeled_field(
    field: FieldDefinition,
    label_to_choice: Dict[str, str],
) -> FieldDefinition:
    if (
        len(label_to_choice) != len(field.choices)
        or set(label_to_choice.values()) != set(field.choices)
    ):
        raise ValueError("Each original choice must appear exactly once.")

    mapping_text = "; ".join(
        f"{json.dumps(label)} means {json.dumps(choice)}"
        for label, choice in label_to_choice.items()
    )
    return FieldDefinition(
        name=field.name,
        description=(
            f"{field.description} Choose the label for the best value. "
            f"Label mapping: {mapping_text}."
        ),
        choices=list(label_to_choice),
    )


multi_token_field = FieldDefinition(
    name="verdict",
    description="The best description of the forecast.",
    choices=["not sunny", "not rainy", "sunny"],
)

collision_context = (
    "The forecast says the sky will stay clear and sunny all day."
)

shuffled_mappings = [
    {"Z": "not sunny", "X": "not rainy", "W": "sunny"},
    {"Q": "not rainy", "W": "not sunny", "Z": "sunny"},
    {"X": "sunny", "Q": "not sunny", "W": "not rainy"},
]

for run_number, label_to_choice in enumerate(shuffled_mappings, start=1):
    labeled_schema = {
        "verdict": make_labeled_field(
            multi_token_field,
            label_to_choice,
        )
    }
    labeled_result = classify_schema(
        context=collision_context,
        schema=labeled_schema,
        temperature=1.0,
    )
    winning_label = labeled_result["predictions"]["verdict"]
    winning_choice = label_to_choice[winning_label]

    print(
        f"Run {run_number}: "
        f"label={winning_label!r}, choice={winning_choice!r}"
    )


## 8. Practical limits

A few details are easy to miss when adapting this pattern:

- **Candidate wording matters.** Sequence likelihood tends to favor shorter or more familiar surface forms. If choices differ greatly in length, test paraphrases or use a separately validated label-based classifier.
- **Letter labels are not automatically neutral.** Asking the model to choose `A`, `B`, or `C` can be faster, but the label token has its own prior. Shuffling label assignments is a useful bias check, not proof that the model understands every choice.
- **The reported probability is local to the list.** Removing or adding a candidate changes the normalization. Evaluate calibration on held-out data before treating it as confidence.
- **Fields are independent here.** For dependent fields, score valid joint assignments or generate them in schema order while carrying the selected values forward.
- **Cache copies trade memory for clarity.** Large candidate sets need batching, a trie with correct full-sequence accounting, or cache objects designed for branching.

The earlier first-token shortcut and greedy trie approach are intentionally omitted. A first-token score is not a complete sequence score, and mixing it with multi-token scores makes the final softmax difficult to interpret.
